# What is LlamaIndex? (And Why It's Not a LangChain Competitor)

LlamaIndex is a framework purpose-built for **RAG** — Retrieval-Augmented Generation. Its job is narrow and specific: connect large language models to _your own data_ by indexing that data and retrieving the right pieces of it at query time.

It's easy to lump LlamaIndex in with LangChain/LangGraph as "yet another LLM framework," but they solve different layers of the same problem:

- **LlamaIndex** — the retrieval layer. Loading, chunking, embedding, indexing, and retrieving your documents.
- **LangChain / LangGraph** — the orchestration layer. Chaining steps together, managing agent state, tool-calling loops, multi-step workflows.

In practice, most production RAG systems use both: LlamaIndex to build and query an index, LangChain/LangGraph to orchestrate how and when that index gets called as part of a larger agent.


Before touching any documents, configure logging and point LlamaIndex at an LLM and an embedding model via the global `Settings` object — every index and query engine in this notebook (and every future one in this series) uses these two models by default.


In [3]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# LlamaIndex and its HTTP client (httpx) log a lot of INFO-level noise by default —
# bump both up to WARNING so only real problems show up in the output below.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Reads the .env file and copies its keys (e.g. OPENAI_API_KEY) into os.environ,
# which is how the OpenAI clients below pick up your API key.
load_dotenv()

# Settings is a global config object — every index and query engine we build in
# this notebook will use these two models unless we override them explicitly.
Settings.llm = OpenAI(model="gpt-4.1-nano")  # the LLM that reads context and writes answers
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")  # turns text into vectors

**The whole pipeline in one shot.** This is the entire RAG loop end-to-end — load documents, build an index, and ask a question — in a handful of lines, wrapped in a `try/except` since network/API calls can fail. We'll slow down and inspect each of these steps individually starting next episode.


In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

try:
    # Load every file in data/sample_docs into Document objects.
    documents = SimpleDirectoryReader("data/sample_docs").load_data()
    # Chunk, embed, and store those documents as a searchable VectorStoreIndex.
    index = VectorStoreIndex.from_documents(documents)
    # Ask a question — this retrieves the most relevant chunks and has the LLM
    # answer using them, all behind one .query() call.
    response = index.as_query_engine().query("What is Naruto's signature technique?")
    print(response)
except Exception as error:
    print(f"Something went wrong: {error}")

Naruto's signature technique is the Rasengan, a swirling ball of concentrated chakra.


### What just happened under the hood?

In those few lines, LlamaIndex did a lot of work for you:

1. **Loaded** every file in `data/sample_docs/` into `Document` objects.
2. **Chunked** each document into smaller `Node` objects, since LLMs can't (and shouldn't) read entire documents on every query.
3. **Embedded** each node into a vector using an OpenAI embedding model, and stored those vectors in memory.
4. **Matched** your question against those vectors to find the most relevant chunks, then handed them to the LLM to generate the final answer.

We'll slow every one of these steps down and look at them individually starting in the next episode.


### Summary

- LlamaIndex specializes in the retrieval layer of RAG — indexing your data and finding the right pieces of it at query time.
- It's not a LangChain competitor; they operate at different layers and are commonly combined (see the capstone episode).
